# Step 10.6 — Model Evaluation and Baseline Comparison

## Objective

Evaluate the fine-tuned SecBERT baseline and the proposed
SecBERT + PPO framework on the same untouched test set.

The evaluation compares:
- Accuracy
- Macro F1
- Weighted F1
- Matthews Correlation Coefficient (MCC)
- Per-class precision, recall, and F1
- Confusion matrices

The test set is used only for final evaluation and is not used
for model selection or training.

## 1. Environment Setup (Google Colab)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a workspace folder in your Drive (change this if you prefer another path)
WORKSPACE = "/content/drive/MyDrive/SOC_Research_Env"
os.makedirs(WORKSPACE, exist_ok=True)
%cd {WORKSPACE}

# Clone the repository if it hasn't been cloned yet, else pull latest changes
if not os.path.exists('soc-agent'):
    !git clone https://github.com/avidzcheetah/soc-agent.git
else:
    !cd soc-agent && git pull

%cd soc-agent

# Install necessary requirements
!pip install -q transformers torch pandas scikit-learn matplotlib seaborn


## 2. Import Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import sys

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the src folder is in the Python path
sys.path.append(os.getcwd())

# Import SecBERT specific components
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader
from src.secbert.dataset import SecBERTDataset

# Import PPO components
from src.ppo.networks import ActorNetwork


## 3. Reproducibility Setup
A fixed seed of 42 is used to support reproducibility of the evaluation procedure.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 4. Define Paths

In [ ]:
TEST_PATH = "data/processed/test.csv"
SECBERT_PATH = "models/secbert_finetuned"
PPO_PATH = "models/ppo_final/best_ppo_policy.pt"
ACTION_SPACE_PATH = "data/action_space.csv"

# Make sure to upload your models to the respective folders in Drive before running the evaluations!
if not os.path.exists(SECBERT_PATH):
    print(f"⚠️ WARNING: SecBERT model not found at {SECBERT_PATH}. Please upload it.")
if not os.path.exists(PPO_PATH):
    print(f"⚠️ WARNING: PPO model not found at {PPO_PATH}. Please upload it.")


## 5. Verify Test Dataset

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test samples: {len(test_df)}")

# Assertions to verify the test dataset structure
assert "text" in test_df.columns
assert "action_label" in test_df.columns

display(test_df.head())


## 6. Define the 20 Action Names

In [ ]:
action_space_df = pd.read_csv(ACTION_SPACE_PATH)
ACTION_NAMES = dict(zip(action_space_df['Index'], action_space_df['Action']))
PHASE_MAP = dict(zip(action_space_df['Index'], action_space_df['CISA Phase']))

for idx, name in ACTION_NAMES.items():
    print(f"{idx}: {name}")


## 7. Evaluate SecBERT Baseline

In [ ]:
# 1. Preprocess text with CISA phase tags (as done during Phase 1 training)
def prepend_phase(df):
    df_copy = df.copy()
    df_copy['text'] = df_copy.apply(lambda row: f"[{PHASE_MAP[row['action_label']]}] {row['text']}", axis=1)
    return df_copy

test_df_prep = prepend_phase(test_df)
test_texts = test_df_prep['text'].tolist()
test_labels = test_df_prep['action_label'].tolist()

# 2. Load Tokenizer and Dataset
tokenizer = AutoTokenizer.from_pretrained(SECBERT_PATH)
test_dataset = SecBERTDataset(test_texts, test_labels, tokenizer, max_length=512)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collator)

# 3. Load Model
secbert_classifier = AutoModelForSequenceClassification.from_pretrained(SECBERT_PATH)
secbert_classifier.to(device)
secbert_classifier.eval()

y_true_secbert = []
y_pred_secbert = []

print("Evaluating SecBERT Baseline...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = secbert_classifier(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)

        y_pred_secbert.extend(preds.cpu().numpy())
        y_true_secbert.extend(labels.cpu().numpy())
        
print("Done!")


In [ ]:
secbert_accuracy = accuracy_score(y_true_secbert, y_pred_secbert)
secbert_macro_f1 = f1_score(y_true_secbert, y_pred_secbert, average="macro", zero_division=0)
secbert_weighted_f1 = f1_score(y_true_secbert, y_pred_secbert, average="weighted", zero_division=0)
secbert_mcc = matthews_corrcoef(y_true_secbert, y_pred_secbert)

print(f"SecBERT Accuracy:    {secbert_accuracy:.4f}")
print(f"SecBERT Macro F1:    {secbert_macro_f1:.4f}")
print(f"SecBERT Weighted F1: {secbert_weighted_f1:.4f}")
print(f"SecBERT MCC:         {secbert_mcc:.4f}")


## 8. Evaluate SecBERT + PPO
Evaluate the already-trained best PPO checkpoint using deterministic evaluation.

In [ ]:
# Note: We will use the underlying SecBERT encoder from the classifier.
encoder_only = getattr(secbert_classifier, secbert_classifier.base_model_prefix)
encoder_only.eval()

# Load PPO policy
state_dim = 768
action_dim = 20

actor = ActorNetwork(state_dim, action_dim).to(device)

# Load the weights (handle different checkpoint saving methods)
checkpoint = torch.load(PPO_PATH, map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and 'actor_state_dict' in checkpoint:
    actor.load_state_dict(checkpoint['actor_state_dict'])
else:
    actor.load_state_dict(checkpoint)
    
actor.eval()

y_true_ppo = []
y_pred_ppo = []

print("Evaluating SecBERT + PPO...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Get embeddings from frozen encoder
        outputs = encoder_only(input_ids, attention_mask=attention_mask)
        # Using CLS token representation to exactly match SecBERTStateEncoder
        embeddings = outputs.last_hidden_state[:, 0, :] 

        # Deterministic action selection (argmax over logits)
        action_logits = actor(embeddings)
        preds = torch.argmax(action_logits, dim=-1)

        y_pred_ppo.extend(preds.cpu().numpy())
        y_true_ppo.extend(labels.cpu().numpy())

print("Done!")


In [ ]:
ppo_accuracy = accuracy_score(y_true_ppo, y_pred_ppo)
ppo_macro_f1 = f1_score(y_true_ppo, y_pred_ppo, average="macro", zero_division=0)
ppo_weighted_f1 = f1_score(y_true_ppo, y_pred_ppo, average="weighted", zero_division=0)
ppo_mcc = matthews_corrcoef(y_true_ppo, y_pred_ppo)

print(f"PPO Accuracy:    {ppo_accuracy:.4f}")
print(f"PPO Macro F1:    {ppo_macro_f1:.4f}")
print(f"PPO Weighted F1: {ppo_weighted_f1:.4f}")
print(f"PPO MCC:         {ppo_mcc:.4f}")


## 9. Main Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Fine-tuned SecBERT", "SecBERT + PPO"],
    "Accuracy": [secbert_accuracy, ppo_accuracy],
    "Macro F1": [secbert_macro_f1, ppo_macro_f1],
    "Weighted F1": [secbert_weighted_f1, ppo_weighted_f1],
    "MCC": [secbert_mcc, ppo_mcc]
})

display(comparison)


In [ ]:
difference = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1", "MCC"],
    "Difference": [
        ppo_accuracy - secbert_accuracy,
        ppo_macro_f1 - secbert_macro_f1,
        ppo_weighted_f1 - secbert_weighted_f1,
        ppo_mcc - secbert_mcc
    ]
})

display(difference)


## 10. Per-Class Comparison
Identifying which response actions PPO improved or degraded.

In [ ]:
secbert_report = classification_report(
    y_true_secbert, y_pred_secbert,
    labels=list(ACTION_NAMES.keys()),
    target_names=list(ACTION_NAMES.values()),
    output_dict=True, zero_division=0
)

ppo_report = classification_report(
    y_true_ppo, y_pred_ppo,
    labels=list(ACTION_NAMES.keys()),
    target_names=list(ACTION_NAMES.values()),
    output_dict=True, zero_division=0
)

records = []
for action_name in ACTION_NAMES.values():
    s_f1 = secbert_report[action_name]['f1-score']
    p_f1 = ppo_report[action_name]['f1-score']
    diff = p_f1 - s_f1
    records.append({
        "Action": action_name,
        "SecBERT F1": s_f1,
        "PPO F1": p_f1,
        "Difference": diff,
        "Support": secbert_report[action_name]['support']
    })

per_class_df = pd.DataFrame(records)
display(per_class_df)


## 11. Confusion Matrices

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(ACTION_NAMES.keys()))
    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(ACTION_NAMES.values()))
    disp.plot(cmap="Blues", ax=ax, xticks_rotation='vertical')
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_cm(y_true_secbert, y_pred_secbert, "Confusion Matrix: Fine-tuned SecBERT")


In [ ]:
plot_cm(y_true_ppo, y_pred_ppo, "Confusion Matrix: SecBERT + PPO")


## 12. Performance Visualizations

In [ ]:
# 1. Overall Metrics Bar Chart
metrics_df = comparison.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x="Metric", y="Score", hue="Model", palette=["#4c72b0", "#dd8452"])
plt.ylim(0, 1.1)
plt.title("Overall Metric Comparison")
plt.legend(loc='lower right')
plt.show()


In [ ]:
# 2. Per-Class F1 Comparison
plt.figure(figsize=(12, 8))
f1_melted = per_class_df[['Action', 'SecBERT F1', 'PPO F1']].melt(id_vars="Action", var_name="Model", value_name="F1 Score")
sns.barplot(data=f1_melted, y="Action", x="F1 Score", hue="Model", palette=["#4c72b0", "#dd8452"], orient='h')
plt.title("Per-Class F1 Score Comparison")
plt.xlim(-0.05, 1.05)
plt.show()


## 13. Test Class Distribution (Imbalance Analysis)

In [ ]:
plt.figure(figsize=(12, 8))
support_df = per_class_df[['Action', 'Support']].sort_values(by="Support", ascending=False)
sns.barplot(data=support_df, y="Action", x="Support", color="#55a868", orient='h')
plt.title("Number of Test Samples per Response Action")
plt.xlabel("Support (Count)")
plt.show()


## Evaluation Summary

The fine-tuned SecBERT baseline and the proposed SecBERT + PPO
framework were evaluated on the same untouched test set.

The comparison considers Accuracy, Macro F1, Weighted F1, and MCC.
Macro F1 is emphasized because the response-action classes are
highly imbalanced.

The results show whether PPO improves overall and class-balanced
response-action selection relative to the SecBERT baseline.